# MMD as a Kernel Mean Embedding

For a signed measure $\xi=\alpha-\beta$ and a positive kernel $k$, the MMD is
$$
\|\xi\|_k=\left\|m_\xi\right\|_{\mathcal H_k},\qquad
m_\xi=\int k(x,\cdot)\,d\xi(x).
$$
This notebook makes this formula concrete.  A one-dimensional empirical signed measure is first shown as a sum of positive red kernel bumps and negative blue kernel bumps.  Then the same samples are embedded in a finite kernel feature sketch, where the MMD is the distance between the two mean embeddings.

In [ ]:
from pathlib import Path
import os
import sys

os.environ.setdefault("MPLCONFIGDIR", "/private/tmp/mpl-ot4ml")

for candidate in [Path.cwd(), Path.cwd() / "notebooks-figures", Path.cwd().parent / "notebooks-figures"]:
    if (candidate / "figure_style.py").exists():
        sys.path.insert(0, str(candidate.resolve()))
        break
else:
    raise RuntimeError("Could not locate figure_style.py")

import matplotlib.pyplot as plt
import numpy as np

from figure_style import (
    BLUE,
    RED,
    VIOLET,
    ORANGE,
    GRAY,
    LIGHT_GRAY,
    DIRAC_MARKER_SIZE,
    box_axes,
    figure_dir,
    interp_color,
    remove_axes,
    save_pdf,
    setup_matplotlib,
)

setup_matplotlib()

NAME = "dualnorms-mmd-kernel-mean-embedding"
OUT = figure_dir(NAME)


## A signed kernel superposition

With a Gaussian kernel, each Dirac mass contributes one bump $k(x_i,\cdot)$.  The witness direction in the RKHS is the signed sum of the red and blue contributions.

In [ ]:
rng = np.random.default_rng(8)
source = np.array([-1.85, -1.22, -0.82, 0.46, 1.10, 1.48])
target = np.array([-1.36, -0.28, 0.18, 0.78, 1.62, 2.04])
sigma = 0.46
z = np.linspace(-2.6, 2.55, 850)

def kernel(u, v, sig=sigma):
    return np.exp(-0.5 * ((u[:, None] - v[None, :]) / sig) ** 2)

Ks = kernel(z, source)
Kt = kernel(z, target)
mean_source = Ks.mean(axis=1)
mean_target = Kt.mean(axis=1)
embedding = mean_source - mean_target
scale = 1.0 / max(abs(embedding).max(), 1e-12)
embedding_plot = scale * embedding
source_plot = scale * mean_source
target_plot = -scale * mean_target

fig, ax = plt.subplots(figsize=(3.6, 1.9))
for j in range(len(source)):
    ax.plot(z, scale * Ks[:, j] / len(source), color=RED, alpha=0.22, lw=0.8)
for j in range(len(target)):
    ax.plot(z, -scale * Kt[:, j] / len(target), color=BLUE, alpha=0.22, lw=0.8)
ax.fill_between(z, 0, np.maximum(embedding_plot, 0), color=RED, alpha=0.18, lw=0)
ax.fill_between(z, 0, np.minimum(embedding_plot, 0), color=BLUE, alpha=0.18, lw=0)
ax.plot(z, embedding_plot, color=VIOLET, lw=1.45)
ax.axhline(0, color=GRAY, lw=0.7, alpha=0.55)
ax.scatter(source, np.full_like(source, 0.08), s=DIRAC_MARKER_SIZE*0.72, color=RED, marker='o', edgecolor='none', zorder=4)
ax.scatter(target, np.full_like(target, -0.08), s=DIRAC_MARKER_SIZE*0.72, color=BLUE, marker='o', edgecolor='none', zorder=4)
ax.set_xlim(z.min(), z.max())
ax.set_ylim(-0.70, 0.70)
ax.set_xlabel(r"$x$")
ax.set_ylabel(r"$m_{\alpha-\beta}(x)$")
box_axes(ax)
fig.tight_layout(pad=0.2)
save_pdf(fig, OUT / 'signed-kernel-sum.pdf')
plt.close(fig)


## A finite feature sketch

The exact RKHS can be infinite-dimensional.  To draw it, we use the top two kernel-principal coordinates of the union of the samples.  The red and blue centers are the empirical feature means, and the violet segment is the MMD vector in this sketch.

In [ ]:
points = np.concatenate([source, target])[:, None]
labels = np.array([0] * len(source) + [1] * len(target))
K = np.exp(-0.5 * ((points - points.T) / sigma) ** 2)
N = len(points)
H = np.eye(N) - np.ones((N, N)) / N
Kc = H @ K @ H
vals, vecs = np.linalg.eigh(Kc)
order = np.argsort(vals)[::-1]
vals = np.maximum(vals[order[:2]], 0)
vecs = vecs[:, order[:2]]
coords = vecs * np.sqrt(vals)[None, :]
coords *= 1.85 / max(np.abs(coords).max(), 1e-12)
mean_red = coords[labels == 0].mean(axis=0)
mean_blue = coords[labels == 1].mean(axis=0)

fig, ax = plt.subplots(figsize=(2.7, 2.35))
ax.plot(coords[:, 0], coords[:, 1], color=LIGHT_GRAY, lw=0.8, alpha=0.8, zorder=0)
ax.scatter(coords[labels == 0, 0], coords[labels == 0, 1], s=DIRAC_MARKER_SIZE*0.78, color=RED, marker='o', edgecolor='none', zorder=3)
ax.scatter(coords[labels == 1, 0], coords[labels == 1, 1], s=DIRAC_MARKER_SIZE*0.78, color=BLUE, marker='o', edgecolor='none', zorder=3)
ax.scatter([mean_red[0]], [mean_red[1]], s=DIRAC_MARKER_SIZE*2.0, color=RED, marker='o', edgecolor='none', zorder=4)
ax.scatter([mean_blue[0]], [mean_blue[1]], s=DIRAC_MARKER_SIZE*2.0, color=BLUE, marker='o', edgecolor='none', zorder=4)
ax.annotate('', xy=mean_blue, xytext=mean_red, arrowprops=dict(arrowstyle='->', lw=1.35, color=VIOLET, shrinkA=2, shrinkB=2))
mid = 0.5 * (mean_red + mean_blue)
ax.text(mid[0] + 0.05, mid[1] + 0.05, r"$\|m_\alpha-m_\beta\|$", color=VIOLET, fontsize=8)
all_pts = np.vstack([coords, mean_red, mean_blue])
pad = 0.18
xmin, ymin = all_pts.min(axis=0); xmax, ymax = all_pts.max(axis=0)
dx, dy = xmax - xmin, ymax - ymin
ax.set_xlim(xmin - pad*dx, xmax + pad*dx)
ax.set_ylim(ymin - pad*dy, ymax + pad*dy)
ax.set_aspect('equal', adjustable='box')
remove_axes(ax)
fig.tight_layout(pad=0.05)
save_pdf(fig, OUT / 'feature-mean-distance.pdf')
plt.close(fig)
